|<img style="float:left;" src="../images/usherb_transp.gif" width=90% height=90%> |<big>Pierre Proulx, ing, professeur</big>|
|:---|:---|
|Département de génie chimique et de génie biotechnologique |** GCH200-Phénomènes d'échanges I **|



### Exercice 6A9, première approche, approximative en utilisant des valeurs moyennes.

In [18]:
import thermo as th
from math import *
import fluids
from fluids.units import *
from thermo.units import Stream 
import numpy as np
from scipy.optimize import fsolve
import sympy as sp

def enleve_unites(liste):      # fonction pour enlever les unités
    l=np.zeros(len(liste))
    for i,t in enumerate(liste):
        l[i]=t.magnitude
    return l
    
def delta_P_Ergun(v0): # la fonction programmée dans fluids est exactement l'équation quadratique d'Ergun du livre.
    print(fluids.dP_packed_bed(Dp, voidage=eps, vs=v0, rho=rho, mu=mu, L=L, Dt=D, 
                            Method='Ergun' ) - difference_de_Pression)
    return fluids.dP_packed_bed(Dp, voidage=eps, vs=v0, rho=rho, mu=mu, L=L, Dt=D, 
                            Method='Ergun' ) - difference_de_Pression
    
# Il suffit de définir les valeurs de Dp, mu, rhp, différence de pression, etc... et de faire résoudre par 
# fsolve pour trouver la vitesse qui satisfait la fonction deltaP=0  qui est définie comme la différence de pression
# dans le lit - la différence de pression imposée de 22 atm.

CO2=th.Chemical('CO2',T=300)
CO2.calculate(P=25*101325)
mu1=CO2.mu
rho1=CO2.rho
CO2.calculate(P=3*101325)
mu2=CO2.mu
rho2=CO2.rho
rho=(rho1+rho2)/2
rho=rho*(u.kg/u.m**3)
mu=(mu1+mu2)/2*u.kg/u.m/u.sec
v0=3*u.m/u.sec
eps=0.41
L=(5.5*u.ft).to(u.m)
D=(4/12*u.ft).to(u.m)
Dp=(1/16*u.inch).to(u.m)
difference_de_Pression=(25*101325-3*101325)*u.Pa
print(difference_de_Pression,L,Dp,D,mu,rho)
difference_de_Pression,L,Dp,D,mu,rho=enleve_unites((difference_de_Pression,L,Dp,D,mu,rho))
v=fsolve(delta_P_Ergun,v0)[0]
print('debit massique en g/sec {:<10.1f}'.format(rho*v*pi*D**2/4*1000))

2229150 pascal 1.6763999999999997 meter 0.0015875 meter 0.10159999999999998 meter 1.5003195769115984e-05 kilogram / meter / second 25.02854728045211 kilogram / meter ** 3
[1357064.07764293]
[1357064.07764293]
[1357064.07764293]
[1357064.18418244]
[128380.83354864]
[14815.65297147]
[206.70511804]
[0.34229928]
[7.93486834e-06]
[-4.65661287e-10]
debit massique en g/sec 479.5     


____
____
____
### Approche plus précise en utilisant la forme différentielle de $$
  \boxed{
  ( \frac  {(\mathscr{P}_0 - \mathscr{P}_L) \rho} {G_0^2}  )( \frac  {D_p} {L}  )( \frac  {\epsilon^3} {1- \epsilon}  )= 150  ( \frac  {1-\epsilon} {D_p G_0 / \mu}  )+\frac {7}{4}
  }
 $$ 
 ### la pression varie en fonction de z, donc la forme différentielle est une variation sur une distance $\Delta z$, on peut donc écrire

### $$ -\frac  {d \mathscr{P}} {dz} = \frac  {150  ( \frac  {1-\epsilon} {D_p G_0 / \mu}  )+\frac {7}{4}} { ( \frac  { \rho} {G_0^2}  )( {D_p}  )( \frac  {\epsilon^3} {1- \epsilon}  ) }$$
### avec la densité variant en fonction de la pression, donc de z. Séparant et intégrant on obtiendra:
### $$ -\int_{25}^{3} \rho d \mathscr{P} = \int_0^L (\frac  {150  ( \frac  {1-\epsilon} {D_p G_0 / \mu}  )+\frac {7}{4}} { ( \frac  { 1} {G_0^2}  )( {D_p}  )( \frac  {\epsilon^3} {1- \epsilon}  ) }) dz $$
### $$ -\int_{25}^{3} \rho d \mathscr{P} =  (\frac  {150  ( \frac  {1-\epsilon} {D_p G_0 / \mu}  )+\frac {7}{4}} { ( \frac  { 1} {G_0^2}  )( {D_p}  )( \frac  {\epsilon^3} {1- \epsilon}  ) }) L $$
____
### On peut facilement intégrer le terme de gauche et solutionner pour trouver $G_0$ maintenant en exprimant la densité en fonction de la pression par la loi des gaz parfaits: $\rho=\frac {PM}{RT}$
### $$-\frac {M}{RT} \int_{25}^{3}P d \mathscr{P} =  (\frac  {150  ( \frac  {1-\epsilon} {D_p G_0 / \mu}  )+\frac {7}{4}} { ( \frac  { 1} {G_0^2}  )( {D_p}  )( \frac  {\epsilon^3} {1- \epsilon}  ) }) L $$
### $$\frac {M(P_0^2-P_L^2)}{2RT}  =  (\frac  {150  ( \frac  {1-\epsilon} {D_p G_0 / \mu}  )+\frac {7}{4}} { ( \frac  { 1} {G_0^2}  )( {D_p}  )( \frac  {\epsilon^3} {1- \epsilon}  ) }) L $$
### On doit maintenant solutionner pour trouver $G_0$ en trouvant les 2 solutions de l'équation quadratique. On utilisera alors sympy


In [21]:
G0=sp.symbols('G_0')
P0=25*101325
PL=3*101325
R=8314.5
gauche=CO2.MW*(P0**2-PL**2)/(2*300*R)
droiten=150*(1-eps)/(Dp*G0/mu)+7/4
droited=1/G0**2*Dp*eps**3/(1-eps)
droite=droiten/droited*L
eq=sp.Eq(gauche,droite)
G0=sp.solve(eq,G0)
print(G0)
G0=G0*u.kg/u.sec/u.m**2
G0=G0.to(u.g/u.cm**2/u.sec)
D=D=(4/12*u.ft).to(u.cm)
w=G0*3.1416*(D)**2/4
print('debit massique {:<5.1f}'.format(w))

[-59.6256177959577, 59.1476757215130]
debit massique [-483.405074137029 479.530235896849] gram / second
